# 🗳️ Parallel Exploration + Ensemble Decision

Several independent agents analyse the same problem **at the same time**, each from a different perspective, and
an aggregator synthesizes their conclusions. The example is a mock **investment committee** answering *"Is NVIDIA
(NVDA) a good long-term investment today?"*:
1. **Bullish Growth Analyst:** innovation, market position, long-term potential
2. **Cautious Value Analyst:** valuation, competition, risks
3. **Quantitative Analyst:** financial metrics and technical indicators only

A **Chief Investment Officer (CIO)** agent then merges the three reports into one structured recommendation.

## Learning Objectives
In this notebook, you will learn:
1. **Independent perspectives** - Give three analysts sharply different personas that never see each other's work
2. **Tool-using analysts** - Run each analyst as a `create_agent` loop that actually executes its searches
3. **Structured aggregation** - Merge the three reports into a Pydantic CIO verdict with a `Literal` recommendation
4. **Fan-out and fan-in** - Start the analysts together from `START` and make the CIO wait for all three
5. **Vote vs verdict** - Compare the CIO's call with the analysts' own parsed votes in code

## Where this fits
Debate (`03_debate.ipynb`) ran perspectives *in turn*, each seeing the others. An ensemble keeps them
**independent and concurrent**, so no analyst anchors on another. This notebook uses LangGraph's parallel execution
without taking it apart: [`01_parallel_agents.ipynb`](../notebooks/05a_parallel_agents.ipynb) in module 09 explains the execution model
behind it (supersteps, joins, reducers and `Send`).

## Prerequisites
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- `TAVILY_API_KEY` in the same `.env` (web search)

> This notebook is for learning agent patterns, not investment advice.

| Property | Value |
|---|---|
| Origin | "Wisdom of crowds" (Surowiecki, 2004); modern LLM ensembles in practice |


### Definition
**Parallel Exploration + Ensemble Decision** is an agentic architecture where a problem is simultaneously processed by multiple independent agents or reasoning paths. The individual outputs are then aggregated, often by a separate agent, through a method like voting, consensus-building, or synthesis to arrive at a final, more robust conclusion.

### High-level Workflow

1.  **Fan-Out (Parallel Exploration):** A user's query is distributed to N independent specialist agents. Crucially, these agents are often given different instructions, personas, or tools to encourage diverse analytical approaches.
2.  **Independent Processing:** Each agent works on the problem in isolation, generating its own complete analysis, conclusion, or answer.
3.  **Fan-In (Aggregation):** The outputs from all N agents are collected.
4.  **Synthesize (Ensemble Decision):** A final "aggregator" or "judge" agent receives all the individual outputs. Its task is to analyze these perspectives, identify common ground, weigh conflicting evidence, and synthesize a comprehensive final answer.

### When to Use / Applications
*   **Hard Reasoning Q&A:** For complex, ambiguous questions where a single line of reasoning might easily miss the nuance (e.g., "What was the primary cause of the 2008 financial crisis?").
*   **Fact-Checking & Verification:** Having multiple agents search for and verify a fact from different sources can drastically reduce hallucinations.
*   **High-Stakes Decision Support:** In fields like medicine or finance, getting a "second opinion" (or third, or fourth) from different AI personas before making a recommendation.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Boosts Reliability & Accuracy:** Averages out the random errors or biases of a single agent, making the final answer much more likely to be correct and well-rounded.
    *   **Reduces Hallucinations:** If one agent hallucinates a fact, the others are unlikely to do the same, and the aggregator can easily spot the outlier.
*   **Weaknesses:**
    *   **Very High Cost:** This is one of the most expensive architectures, as it multiplies the number of LLM calls by the number of agents in the ensemble (plus the final aggregation call).
    *   **Increased Latency:** The system must wait for all parallel paths to complete before the final synthesis can begin.

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and `TAVILY_API_KEY` and imports the building blocks.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import operator
import re
import time
from collections import Counter
from typing import Annotated, Any, Dict, List, Literal, Optional

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from typing_extensions import TypedDict

console = Console()
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the three analysts and the CIO
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0.3)

print(f"✅ LLM initialized: {llm.model_name}")

---
## 👥 Part 1: The Diverse Specialist Analysts

Cognitive diversity is what makes an ensemble useful, so each analyst gets a sharply different persona and its own
`create_agent` search loop. All three write their report to the same `analyses` key during the same step, so the key
carries a reducer: `operator.or_` merges the three one-entry dicts. Why parallel writes to one key need a reducer is
covered in module 09; here it is one line. Each analyst also records when it started and finished, for the one-line
overlap check in Part 5.

In [ ]:
# ============================================================================
# ANALYST STATE AND NODES
# ============================================================================
search_tool = TavilySearch(max_results=3)


class EnsembleState(TypedDict):
    query: str
    analyses: Annotated[Dict[str, str], operator.or_]   # three parallel writers: merge their dicts
    timings: Annotated[Dict[str, tuple], operator.or_]  # (start, end) per analyst, for the overlap check
    final_recommendation: Optional[Any]


RUN_START = time.monotonic()


def create_analyst_node(persona: str, agent_name: str):
    system_prompt = (
        f"You are an expert financial analyst. Your persona is '{persona}'. Use your search tool to gather "
        "up-to-date information, then give a concise investment analysis (at most 250 words). "
        "End with exactly two lines:\nRecommendation: <Buy|Hold|Sell>\nConfidence: <1-10>"
    )
    agent = create_agent(llm, tools=[search_tool], system_prompt=system_prompt)

    def analyst_node(state: EnsembleState) -> Dict[str, Any]:
        start = time.monotonic() - RUN_START
        console.print(f"--- 👨‍💻 {agent_name} started at t={start:.1f}s ---")
        result = agent.invoke({"messages": [HumanMessage(content=state["query"])]}, {"recursion_limit": 12})
        end = time.monotonic() - RUN_START
        console.print(f"--- ✅ {agent_name} finished at t={end:.1f}s ---")
        return {"analyses": {agent_name: result["messages"][-1].content}, "timings": {agent_name: (start, end)}}

    return analyst_node


bullish_analyst_node = create_analyst_node(
    "The Bullish Growth Analyst: optimistic about technology and innovation; focuses on TAM, leadership, "
    "technological moats and long-term growth, and downplays short-term volatility.",
    "BullishAnalyst",
)
value_analyst_node = create_analyst_node(
    "The Cautious Value Analyst: a skeptic focused on fundamentals and risk; scrutinizes valuation multiples, "
    "debt, competitive threats and bubble risk.",
    "ValueAnalyst",
)
quant_analyst_node = create_analyst_node(
    "The Quantitative Analyst: purely data-driven; reports revenue growth, EPS, margins, valuation multiples "
    "and technical indicators, and ignores narratives.",
    "QuantAnalyst",
)
print("✅ Three analyst nodes defined")

---
## 🏛️ Part 2: The CIO Aggregator

The CIO reads all three reports and returns a structured `FinalRecommendation`. The recommendation field is a
`Literal`, so it can be compared with the analysts' votes in code.

In [ ]:
# ============================================================================
# CIO AGGREGATOR NODE
# ============================================================================
class FinalRecommendation(BaseModel):
    """The final, synthesized investment thesis from the CIO."""
    final_recommendation: Literal["Strong Buy", "Buy", "Hold", "Sell", "Strong Sell"]
    confidence_score: float = Field(description="Confidence in this recommendation, from 1.0 to 10.0.")
    synthesis_summary: str = Field(description="How the analysts' views were weighed: agreement and contention.")
    identified_opportunities: List[str] = Field(description="The primary opportunities or bullish points.")
    identified_risks: List[str] = Field(description="The primary risks or bearish points.")


cio_llm = llm.with_structured_output(FinalRecommendation)


def cio_synthesizer_node(state: EnsembleState) -> Dict[str, Any]:
    console.print("--- 🏛️ CIO synthesizing the committee's reports ---")
    all_analyses = "\n\n---\n\n".join(f"**Analysis from {name}:**\n{text}" for name, text in state["analyses"].items())
    decision = cio_llm.invoke(
        "You are the Chief Investment Officer of a major fund. Synthesize your analysts' diverse, often conflicting "
        "reports into one balanced, actionable thesis, weighing growth potential against risks and valuation.\n\n"
        f"Query: {state['query']}\n\n{all_analyses}"
    )
    return {"final_recommendation": decision}


print("✅ CIO aggregator defined")

---
## 🏗️ Part 3: Fan-Out and Fan-In

One edge from `START` to each analyst runs them in the same superstep (in parallel). The list edge
`[analysts] -> cio_synthesizer` waits for all three before the CIO runs. Supersteps and joins are explained
in module 09's `01_parallel_agents.ipynb`.

In [ ]:
# ============================================================================
# ASSEMBLE THE ENSEMBLE GRAPH
# ============================================================================
ANALYSTS = ["bullish_analyst", "value_analyst", "quant_analyst"]

workflow = StateGraph(EnsembleState)
workflow.add_node("bullish_analyst", bullish_analyst_node)
workflow.add_node("value_analyst", value_analyst_node)
workflow.add_node("quant_analyst", quant_analyst_node)
workflow.add_node("cio_synthesizer", cio_synthesizer_node)

for name in ANALYSTS:
    workflow.add_edge(START, name)            # fan-out
workflow.add_edge(ANALYSTS, "cio_synthesizer")  # fan-in: waits for all three
workflow.add_edge("cio_synthesizer", END)

ensemble_agent = workflow.compile()
print("✅ Ensemble graph compiled. Edges:")
for edge in ensemble_agent.get_graph().edges:
    print(f"  {edge.source} -> {edge.target}{'  (conditional)' if edge.conditional else ''}")

---
## ▶️ Part 4: Running the Committee

Each analyst prints when it starts and when it finishes, so you can watch the three loops overlap; Part 5 checks the
overlap in code. The three reports and the CIO's thesis follow.

In [ ]:
# ============================================================================
# RUN THE INVESTMENT COMMITTEE
# ============================================================================
query = "Based on recent news, financial performance, and future outlook, is NVIDIA (NVDA) a good long-term investment today?"
RUN_START = time.monotonic()
result = ensemble_agent.invoke({"query": query, "analyses": {}, "timings": {}})
total = time.monotonic() - RUN_START

for name, analysis in result["analyses"].items():
    console.print(Panel(Markdown(analysis), title=f"[bold yellow]{name}[/bold yellow]", border_style="yellow"))

rec = result["final_recommendation"]
opportunities = "\n".join(f"* {o}" for o in rec.identified_opportunities)
risks = "\n".join(f"* {r}" for r in rec.identified_risks)
console.print(Panel(
    f"[bold]Final Recommendation:[/bold] {rec.final_recommendation}\n"
    f"[bold]Confidence Score:[/bold] {rec.confidence_score}/10\n\n"
    f"[bold]Synthesis Summary:[/bold]\n{rec.synthesis_summary}\n\n"
    f"[bold]Identified Opportunities:[/bold]\n{opportunities}\n\n"
    f"[bold]Identified Risks:[/bold]\n{risks}",
    title="[bold green]Chief Investment Officer's Thesis[/bold green]",
    border_style="green",
))

---
## 🗳️ Part 5: Vote vs Verdict

Two checks, computed rather than assumed:
- **The analysts overlapped**: one line from the recorded start and end times
- **Vote vs verdict**: each analyst ended its report with `Recommendation: Buy|Hold|Sell`. The cell parses those
  votes, finds the majority (if there is one) and sets it against the CIO's structured `final_recommendation`

In [ ]:
# ============================================================================
# OVERLAP AND VOTE VS VERDICT
# ============================================================================
windows = result["timings"].values()
overlap = max(start for start, _ in windows) < min(end for _, end in windows)
print(f"Analyst windows overlap: {'✅' if overlap else '❌'} (wall time {total:.1f}s for three analyst loops plus the CIO)")

votes = {}
for name, text in result["analyses"].items():
    m = re.search(r"Recommendation:\s*\**\s*(Buy|Hold|Sell)", text, re.I)
    votes[name] = m.group(1).title() if m else "unparsed"
print("\nAnalyst votes:", votes)
print("CIO verdict:  ", rec.final_recommendation)

top, count = Counter(votes.values()).most_common(1)[0]
if count < 2 or top == "unparsed":
    print("No majority among the analysts: the CIO had to weigh a split committee")
else:
    cio_direction = rec.final_recommendation.split()[-1]  # "Strong Buy" -> "Buy"
    relation = "✅ agrees with" if cio_direction == top else "↔️ departs from"
    print(f"Majority vote: {top} ({count}/{len(votes)}). The CIO {relation} the majority")

### Reading the results
1. **Overlap**: ✅ means all three analyst windows were open at the same time, so the three search loops ran
   concurrently. That concurrency is where the pattern's latency cost is recovered.
2. **Diversity**: the three reports, and often the three votes, differ. A single agent with a neutral prompt would
   produce one of these angles, not all three.
3. **Vote vs verdict**: the CIO may agree with the majority or depart from it. Either is legitimate as long as its
   `synthesis_summary` explains why; a CIO that departs from the majority with no reason in its summary is the
   failure to look for.

The cost is real: every analyst is its own agent loop with searches, plus the aggregation call.

---
## 📝 Summary

- **Independent perspectives**: three personas that never see each other's work, so none anchors on another.
- **Tool-using analysts** run as `create_agent` loops, so their searches actually execute.
- **Structured aggregation**: a `Literal` recommendation lets code set the ensemble's verdict against its members' votes.
- **Fan-out / fan-in**: edges from `START` start the analysts together, and a list edge makes the CIO wait for all
  three. The shared `analyses` key needs a reducer (`operator.or_`).
- The execution model under all of this (supersteps, joins, reducers and `Send`) is the subject of module 09's
  `01_parallel_agents.ipynb`.

### Next Steps
- `07_cellular_automata.ipynb`: many *very* simple agents on a grid, with no controller at all